# Estimering af et boligenergiefterspørgselssystem med Seemingly Unrelated Regression

## Sammenfatning for ledelsen

Et regionalt forsyningsselskab estimerer et **energiefterspørgselssystem** for boliger med to ligninger — budgetandele for elektricitet og naturgas som funktioner af egenpris, krydspris og husstandsindkomst — ved hjælp af **PROC SYSLIN** med metoden seemingly-unrelated-regression (SUR). De to andelsligninger deler et fælles husstandsbudget, så deres fejlled er krydskorrelerede; SUR estimerer ligningerne i fællesskab for at udnytte den korrelation og genfinder fortegnskohærente egen- og krydspriseffekter samt leverer den krydsligningskovarians og de restriktionstest, en efterspørgselsanalytiker har brug for.

## Datakilder

| Datasæt | Rækker | Granularitet | Nøglevariabler | Beskrivelse |
|---------|------|-------|---------------|-------------|
| `energy` | 60 | én række pr. månedlig markedsobservation | `month`, `lp_elec`, `lp_gas`, `lincome`, `w_elec`, `w_gas` | Syntetisk månedligt panel for boligenergimarkedet. `lp_elec`/`lp_gas` er logaritmiske realpriser på elektricitet og naturgas; `lincome` er logaritmisk real husstandsindkomst; `w_elec`/`w_gas` er udgiftsbudgetandele for elektricitet og naturgas, genereret ud fra en kendt AIDS-lignende efterspørgselsstruktur plus korreleret krydsligningsstøj. |

# Estimering af et boligenergiefterspørgselssystem med Seemingly Unrelated Regression

Et regionalt gas- og elforsyningsselskab ønsker at forstå, hvordan husstande omfordeler deres energibudget mellem **elektricitet** og **naturgas**, når relative priser og indkomst ændrer sig. Den naturlige ramme er et **efterspørgselssystem**: et sæt budgetandelsligninger estimeret i fællesskab.

To forhold gør fælles estimering til det rette værktøj:

- Andelsligningerne trækker på et fælles husstandsbudget, så deres fejlled er **krydskorrelerede** — når en husstand bruger mere på elektricitet, bruger den mindre på gas. At estimere ligningerne sammen med **seemingly unrelated regression (SUR)** udnytter den korrelation til effektivitet.
- Økonomisk teori pålægger **lineære restriktioner** (adding-up, homogenitet, symmetri), som en systemestimator kan håndhæve direkte.

`PROC SYSLIN` med `SUR`-optionen er bygget netop til denne situation. Den tilpasser hver andelsligning, estimerer krydsligningsfejlkovariansen og gen-tilpasser systemet med den kovarians for at levere effektive, teorikohærente elasticiteter — sammen med krydsmodelkovariansmatricen og fælles restriktionstest.

I denne notebook vil vi:

1. Generere et syntetisk månedligt energimarkedspanel ud fra en kendt andelsstruktur.
2. Tilpasse et andelssystem med to ligninger med SUR.
3. Sammenligne den fælles SUR-tilpasning med ligning-for-ligning-OLS.
4. Pålægge en homogenitetsrestriktion og aflæse dens fælles F-test.
5. Udtrække koefficienttabellen til elasticitetsrapportering.

## Trin 1 — Generér et syntetisk energimarkedspanel

Vi simulerer 60 månedlige observationer af et lille **energiefterspørgselssystem med to goder** (elektricitet og naturgas). Den datagenererende proces er en lineariseret, AIDS-lignende budgetandelsmodel:

```
w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1
w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2
```

Vi indbygger bevidst **krydsligningskorrelation**: når husstande bruger mere på elektricitet, bruger de mindre på gas, så `e1` og `e2` er negativt korrelerede. En fælles energimarkedsprisfaktor driver også begge logaritmiske priser sammen. Det er disse forhold, der belønner fælles SUR-estimering frem for at tilpasse hver ligning for sig.

In [1]:
data energy;
   CALL streaminit(70123);

   /* True structural coefficients (linearized AIDS share system) */
   a1  = 0.55;  g11 =  0.12; g12 = -0.08; b1 = -0.030;
   a2  = 0.45;  g21 = -0.08; g22 =  0.10; b2 = -0.025;

   GØR month = 1 TIL 60;
      /* A common energy-market price factor drives BOTH prices,
         creating the collinearity that makes the problem ill-posed. */
      energy_factor = rand('NORMAL', 0, 0.15);

      lp_elec = 4.6 + energy_factor + rand('NORMAL', 0, 0.05);
      lp_gas  = 4.2 + energy_factor + rand('NORMAL', 0, 0.05);
      lincome = 10.8 + 0.004*month + rand('NORMAL', 0, 0.06);

      /* Negatively correlated cross-equation errors (budget tradeoff) */
      shock = rand('NORMAL', 0, 0.010);
      e1 =  shock + rand('NORMAL', 0, 0.006);
      e2 = -shock + rand('NORMAL', 0, 0.006);

      w_elec = a1 + g11*lp_elec + g12*lp_gas + b1*lincome + e1;
      w_gas  = a2 + g21*lp_elec + g22*lp_gas + b2*lincome + e2;

      UDDATA;
   SLUT;

   BEHOLD month lp_elec lp_gas lincome w_elec w_gas;
KØR;

PROCEDURE GENNEMSNIT data=energy n mean std MIN MAX maxdec=3;
   VARIABEL w_elec w_gas lp_elec lp_gas lincome;
KØR;

                                                  The MEANS Procedure

 Variable        N           Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------
 W_ELEC         60          0.440       0.011       0.417       0.464
 W_GAS          60          0.228       0.014       0.198       0.256
 LP_ELEC        60          4.617       0.142       4.354       4.911
 LP_GAS         60          4.211       0.162       3.818       4.566
 LINCOME        60         10.916       0.088      10.723      11.174
 --------------------------------------------------------------------



NOTE: DATA energy


NOTE: Wrote energy (60 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Trin 2 — Baseline-SUR-estimering af efterspørgselssystemet

Vi estimerer begge andelsligninger i fællesskab. `SUR`-optionen fortæller `PROC SYSLIN`, at den skal estimere krydsligningsfejlkovariansen og bruge den til en effektiv feasible-GLS-tilpasning. To mærkede `MODEL`-sætninger (`elec` og `gas`) definerer systemet; hver regresserer en budgetandel på de to logaritmiske priser og logaritmisk indkomst.

SYSLIN rapporterer hver lignings parameterestimater og, til sidst, **Cross-Model Covariance Matrix** — den estimerede kovarians mellem de to ligningers fejlled, som SUR udnytter.

In [2]:
PROCEDURE syslin data=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
KØR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Trin 3 — Sammenlign med ligning-for-ligning-OLS

For at se, hvad SUR giver os, gen-tilpasser vi de samme to ligninger med almindelig mindste kvadraters metode (standardmetoden, én ligning ad gangen). OLS ignorerer krydsligningsfejlkorrelationen, så den er konsistent, men mindre effektiv end SUR, når den korrelation er forskellig fra nul — som den er her ved konstruktion.

At sammenligne de to koefficienttabeller viser, hvordan estimaterne og deres standardfejl forskydes, når systemstrukturen tages i betragtning.

In [3]:
PROCEDURE syslin data=energy ols;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
KØR;


                       The SYSLIN Procedure

                  OLS Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.155544       5.15      0.0000
  LP_ELEC        1      0.112463    0.021989       5.11      0.0000
  LP_GAS         1     -0.097938    0.019300      -5.07      0.0000
  LINCOME        1     -0.042842    0.013702      -3.13      0.0028

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (OLS)
NOTE: PROC SYSLIN completed.


## Trin 4 — Pålæg økonomisk teori og test den

Efterspørgselsteori siger, at nominelle pris-/indkomsteffekter bør overholde **homogenitet af grad nul**: at skalere alle priser og indkomst lader den reale efterspørgsel være uændret, så pris- og indkomstkoefficienterne inden for en ligning summer til nul. Vi pålægger det på elektricitetsandelen med en `RESTRICT`-sætning.

SYSLIN gen-tilpasser systemet underlagt begrænsningen og rapporterer en **Restriction Results**-F-test af, om restriktionen er i overensstemmelse med data — en direkte, fælles test af homogenitetshypotesen.

In [4]:
PROCEDURE syslin data=energy ON;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;

   /* Homogeneity on the electricity share equation:
      price and income coefficients sum to zero. */
   restrict lp_elec + lp_gas + lincome = 0;
KØR;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: PROC SYSLIN completed.


## Trin 5 — Opsaml estimater til elasticitetsrapportering

Til en endelig rapport gemmer vi de konvergerede koefficienter med `OUTEST=`. Det resulterende datasæt bærer én række pr. ligning med skæringspunkt- og hældningsestimater plus tilpasningsstatistikker, som føder forsyningsselskabets egen- og krydspriselasticitetsberegninger ved andele på stikprøvemiddel. Vi udskriver det for en ordens skyld.

In [5]:
PROCEDURE syslin data=energy ON outest=demand_est;
   elec: MODEL w_elec = lp_elec lp_gas lincome;
   gas:  MODEL w_gas  = lp_elec lp_gas lincome;
KØR;

PROCEDURE UDSKRIV data=demand_est noobs;
   TITEL "SUR demand-system coefficient estimates";
KØR;
TITEL;


                       The SYSLIN Procedure

                  SUR Estimation

  Model elec Dependent Variable: W_ELEC

  Number of Observations                       60
  SSE                                      0.0048
  MSE                                    0.000085
  R-Square                                 0.3865
  Adj R-Sq                                 0.3537

                      Parameter Estimates

                                   Standard
  Variable     DF     Estimate        Error    t Value    Pr > |t|
  ----------  ---  ------------  ----------  ---------  ----------
  Intercept      1      0.801017    0.150270       5.33      0.0000
  LP_ELEC        1      0.112463    0.021244       5.29      0.0000
  LP_GAS         1     -0.097938    0.018646      -5.25      0.0000
  LINCOME        1     -0.042842    0.013238      -3.24      0.0020

  Model gas Dependent Variable: W_GAS

  Number of Observations                       60
  SSE                                      0.

NOTE: PROC SYSLIN data=energy

NOTE: Using Python for simultaneous equations estimation (SUR)
NOTE: Wrote OUTEST= dataset demand_est (2 rows).
NOTE: PROC SYSLIN completed.
NOTE: PROC PRINT data=demand_est

NOTE: PROC PRINT completed: 2 observations printed, 12 variables


## Fortolkning af resultaterne

**Fortegnskohærens.** SUR-estimaterne genfinder den substitutionsstruktur, der er indbygget i data. I elektricitetsandelsligningen er **egen-log-priskoefficienten positiv** (`LP_ELEC` = 0.112), mens **krydspriskoefficienten er negativ** (`LP_GAS` = -0.098); i gasandelsligningen spejles mønsteret (egen `LP_GAS` = 0.114, kryds `LP_ELEC` = -0.075). Hver egen- og krydspriseffekt er statistisk signifikant (hver `Pr > |t|` under 0.005), så substitutionsfortegnene er fast identificeret frem for artefakter af en støjfyldt tilpasning.

**SUR udnytter krydsligningskorrelationen.** **Cross-Model Covariance Matrix** viser en negativ off-diagonal (-0.000068): husstande veksler elektricitetsforbrug mod gasforbrug, præcis som konstrueret. Fordi den kovarians er forskellig fra nul, er fælles SUR-estimering mere effektiv end at tilpasse hver ligning for sig — SUR-standardfejlene i Trin 2 er gennemgående en anelse mindre end ligning-for-ligning-OLS-standardfejlene i Trin 3 (for eksempel falder gassens `LP_ELEC`-standardfejl fra 0.0264 under OLS til 0.0255 under SUR), mens punktestimaterne er uændrede. Den strammere inferens er udbyttet ved at modellere systemet frem for to separate regressioner.

**Teorien holder.** At pålægge **homogenitet af grad nul** på elektricitetsandelen (pris- og indkomstkoefficienter summer til nul) via `RESTRICT` gav en Restriction Results-F-test på 2.14 med `Pr > F` = 0.149. Restriktionen **forkastes ikke**, så teoriens homogenitetsforudsigelse er i overensstemmelse med denne stikprøve — forsyningsselskabet kan trygt tage de begrænsede, teorikohærente estimater med i rapporteringen.

**Operationel anvendelse.** `OUTEST=`-tabellen gemmer én række pr. ligning med skæringspunkt- og hældningsestimater og tilpasningsstatistikker. Disse koefficienter omregnes direkte til egen- og krydspriselasticiteter og en indkomst-(udgifts-)elasticitet ved andele på stikprøvemiddel (`W_ELEC` ~ 0.44, `W_GAS` ~ 0.23 fra Trin 1), hvilket giver forsyningsselskabet et forsvarligt, teorikonsistent grundlag for efterspørgselsprognoser og takstdesignscenarier.